# **TFT**

Temporal Fusion Transformer (TFT) — это архитектура для прогнозирования временных рядов, которая:

комбинирует LSTM + Transformers (attention);

умеет работать с:

историческими признаками (прошлые значения),

показателями в будущих датах (календарь, планы, расписание),

статическими признаками (тип инструмента, параметры стратегии и т.п.);

специально построена так, чтобы быть интерпретируемой:

какие фичи важнее,

какие моменты времени важнее.

Attention(Q, K, V) = softmax(QKᵀ / √dₖ) · V


Q — query; K — key; V — value; dₖ — размерность ключей.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LSTM, LayerNormalization,
    MultiHeadAttention, Add, Lambda
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
df = pd.read_csv("Brent.csv")
df

,DateTime,Open,High,Low,Close,Alligator_Jaw,Alligator_Teeth,Alligator_Lips,Fractal_Up,Fractal_Down,...,AO_saucer_down,EntrySignal,EntryReason,Fractal_Up_conf,Fractal_Down_conf,AddOn_Anchor_Level,AddOn_Anchor_IsUp,AddOn_Size_Pct,AddOn_Ready,AddOn_Triggered
0,2015-10-26 10:00:00,48.05,48.12,47.89,48.09,48.28815,48.15758,48.06107,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
1,2015-10-26 11:00:00,48.10,48.36,48.00,48.30,48.26098,48.12601,48.05886,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
2,2015-10-26 12:00:00,48.30,48.35,48.18,48.30,48.23552,48.11588,48.05209,0,0,...,0,-1,three_color,0,0,NaN,NaN,NaN,0,0
3,2015-10-26 13:00:00,48.30,48.34,48.05,48.09,48.21971,48.10765,48.04267,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
4,2015-10-26 14:00:00,48.11,48.28,47.97,48.07,48.19550,48.09732,48.07014,0,0,...,0,0,NaN,0,0,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36239,2025-10-21 23:00:00,61.55,61.73,61.55,61.65,61.17612,61.14427,61.29423,0,0,...,0,0,NaN,0,0,61.52,1.0,0.3,0,0
36240,2025-10-22 09:00:00,62.27,62.66,62.26,62.46,61.16565,61.21749,61.31439,1,0,...,0,0,NaN,0,0,61.52,1.0,0.3,0,0
36241,2025-10-22 10:00:00,62.45,62.49,62.19,62.35,61.13752,61.23530,61.35151,0,0,...,0,0,NaN,0,0,61.52,1.0,0.3,0,0
36242,2025-10-22 11:00:00,62.35,62.55,62.25,62.36,61.14002,61.25526,61.40921,0,0,...,0,0,NaN,1,0,61.52,1.0,0.3,0,0


In [ ]:
df = pd.read_csv("Brent.csv")
df = df.sort_values("DateTime").reset_index(drop=True)

# список колонок, где у тебя бывают NaN
na_cols = ["AddOn_Anchor_Level", "AddOn_Anchor_IsUp", "AddOn_Size_Pct"]

# оставляем только существующие (вдруг в другом инструменте их нет)
na_cols = [c for c in na_cols if c in df.columns]

# чистим строки, где в этих колонках NaN
df = df.dropna(subset=na_cols).reset_index(drop=True)

print("NaN после чистки:")
print(df[na_cols].isna().sum())


NaN после чистки:
AddOn_Anchor_Level    0
AddOn_Anchor_IsUp     0
AddOn_Size_Pct        0
dtype: int64


In [ ]:
scale_cols = [
    "Open", "High", "Low", "Close",
    "Alligator_Jaw", "Alligator_Teeth", "Alligator_Lips",
    "AO",
    "AddOn_Anchor_Level", "AddOn_Size_Pct"
]

scaler = RobustScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])


In [ ]:
columns = [
    "AddOn_Anchor_Level",
    "AddOn_Anchor_IsUp",
    "AddOn_Size_Pct"
]

df = df.dropna(subset=columns).reset_index(drop=True)

In [ ]:
# 1) Загружаем данные
df = pd.read_csv("Brent.csv")

# 2) Очень важно: сортируем по времени, чтобы shift(-H) смотрел ВПЕРЁД
df = df.sort_values("DateTime").reset_index(drop=True)

H = 20  # горизонт сделки (баров) — можно менять

def add_goodtrade_target(df, horizon=20):
    df = df.copy()

    # будущая цена через H баров
    df["Close_fwd"] = df["Close"].shift(-horizon)

    # доходность по направлению сигнала
    ret_long  = (df["Close_fwd"] - df["Close"]) / df["Close"]
    ret_short = (df["Close"] - df["Close_fwd"]) / df["Close"]

    df["ret_H"] = np.where(
        df["EntrySignal"] > 0,  ret_long,
        np.where(df["EntrySignal"] < 0, ret_short, 0.0)
    )

    # таргет: есть сигнал и ret_H > 0
    df["GoodTrade"] = ((df["EntrySignal"] != 0) & (df["ret_H"] > 0)).astype(int)

    # убираем хвост, где нет Close_fwd
    df = df.iloc[:-horizon].reset_index(drop=True)
    return df

df = add_goodtrade_target(df, horizon=H)

print("'GoodTrade' в колонках:", "GoodTrade" in df.columns)
print(df[["DateTime", "Close", "Close_fwd", "EntrySignal", "ret_H", "GoodTrade"]].head())



'GoodTrade' в колонках: True
              DateTime  Close  Close_fwd  EntrySignal     ret_H  GoodTrade
0  2015-10-26 10:00:00  48.09      46.70            0  0.000000          0
1  2015-10-26 11:00:00  48.30      46.77            0  0.000000          0
2  2015-10-26 12:00:00  48.30      46.81           -1  0.030849          1
3  2015-10-26 13:00:00  48.09      46.58            0  0.000000          0
4  2015-10-26 14:00:00  48.07      46.77            0  0.000000          0


In [ ]:
# Берём только числовые колонки
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Что точно НЕ должно попасть в признаки (будущее + таргет)
drop_feature_cols = ["Close_fwd", "ret_H", "GoodTrade"]

feature_cols = [c for c in numeric_cols if c not in drop_feature_cols]
print("Фичи для TFT-style модели:", feature_cols)

# Сплит по времени: первые 80% - train, остальное - test
split_bar = int(len(df) * 0.8)

train_df = df.iloc[:split_bar].copy()
test_df  = df.iloc[split_bar:].copy()

print("Train bars:", len(train_df), "Test bars:", len(test_df))
print("Доля GoodTrade=1 (train):", train_df["GoodTrade"].mean())
print("Доля GoodTrade=1 (test) :", test_df["GoodTrade"].mean())


Фичи для TFT-style модели: ['Open', 'High', 'Low', 'Close', 'Alligator_Jaw', 'Alligator_Teeth', 'Alligator_Lips', 'Fractal_Up', 'Fractal_Down', 'AO', 'Color AO', 'Alligator_Bullish', 'Alligator_Bearish', 'AlligatorStart_Long', 'AlligatorStart_Short', 'AO_sign', 'AO_zero_up', 'AO_zero_down', 'AO_three_green', 'AO_three_red', 'AO_saucer_up', 'AO_saucer_down', 'EntrySignal', 'Fractal_Up_conf', 'Fractal_Down_conf', 'AddOn_Anchor_Level', 'AddOn_Anchor_IsUp', 'AddOn_Size_Pct', 'AddOn_Ready', 'AddOn_Triggered']
Train bars: 28979 Test bars: 7245
Доля GoodTrade=1 (train): 0.02425894613340695
Доля GoodTrade=1 (test) : 0.023740510697032435


In [ ]:
from sklearn.preprocessing import StandardScaler

# Масштабируем признаки: fit на train, transform на train и test
scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols]  = scaler.transform(test_df[feature_cols])

SEQ_LEN = 50  # длина окна истории (можно 30, 50, 100 — потом поиграешься)

def make_sequences(df_part, feature_cols, seq_len=50):
    data = df_part[feature_cols].values
    targets = df_part["GoodTrade"].values
    signals = df_part["EntrySignal"].values

    X_list, y_list = [], []

    for i in range(seq_len - 1, len(df_part)):
        # берём только те точки, где на последнем баре окна есть сигнал
        if signals[i] == 0:
            continue

        X_seq = data[i - seq_len + 1 : i + 1, :]  # [seq_len, num_features]
        y_val = targets[i]

        X_list.append(X_seq)
        y_list.append(y_val)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)
    return X, y

X_train, y_train = make_sequences(train_df, feature_cols, seq_len=SEQ_LEN)
X_test, y_test   = make_sequences(test_df,  feature_cols, seq_len=SEQ_LEN)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("Доля GoodTrade=1 в train seq:", (y_train == 1).mean())
print("Доля GoodTrade=1 в test seq :", (y_test == 1).mean())


X_train: (28930, 50, 30) X_test: (7196, 50, 30)
Доля GoodTrade=1 в train seq: 0.024230902177670238
Доля GoodTrade=1 в test seq : 0.02376320177876598


In [ ]:
time_steps = SEQ_LEN
num_features = X_train.shape[2]

inp = Input(shape=(time_steps, num_features))

# 1. Проекция фич (на всякий случай)
x = Dense(64, activation="relu")(inp)

# 2. LSTM-энкодер (локальные паттерны)
x = LSTM(64, return_sequences=True)(x)  # (batch, T, 64)

# 3. Multi-head self-attention (temporal)
attn_output = MultiHeadAttention(
    num_heads=4,
    key_dim=64,
    dropout=0.1
)(x, x)  # self-attention: query=x, key=x, value=x

# Residual + LayerNorm
x1 = Add()([x, attn_output])
x1 = LayerNormalization(epsilon=1e-6)(x1)

# 4. Position-wise feed-forward (FFN)
ffn = Dense(128, activation="relu")(x1)
ffn = Dropout(0.1)(ffn)
ffn = Dense(64, activation="relu")(ffn)

# Residual + LayerNorm
x2 = Add()([x1, ffn])
x2 = LayerNormalization(epsilon=1e-6)(x2)

# 5. Берём представление последнего шага по времени
x_last = Lambda(lambda t: t[:, -1, :])(x2)  # (batch, 64)

# 6. Классификатор
x_last = Dense(32, activation="relu")(x_last)
x_last = Dropout(0.3)(x_last)
out = Dense(1, activation="sigmoid")(x_last)

model = Model(inputs=inp, outputs=out)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 50, 30)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 50, 64)    │      1,984 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 50, 64)    │     33,024 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 50, 64)    │     66,368 │ lstm_1[0][0],     │
│ (MultiHeadAttentio… │                   │            │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 50, 64)    │          0 │ lstm_1[0][0],     │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 64)    │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 50, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 50, 128)   │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 50, 64)    │      8,256 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 50, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 64)    │        128 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 64)        │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 32)        │      2,080 │ lambda_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 32)        │          0 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 1)         │         33 │ dropout_5[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 120,321 (470.00 KB)

 Trainable params: 120,321 (470.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Посмотрим дисбаланс
pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
print("pos:", pos, "neg:", neg)

# Вариант: использовать веса классов
scale_pos = neg / pos if pos > 0 else 1.0
class_weight = {0: 1.0, 1: scale_pos}
print("class_weight:", class_weight)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    class_weight=class_weight,   # если хочешь без весов — просто убери этот аргумент
    verbose=1
)


pos: 701 neg: 28229
class_weight: {0: 1.0, 1: np.float64(40.269614835948644)}
Epoch 1/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 90s 177ms/step - accuracy: 0.8848 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 2/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 80s 177ms/step - accuracy: 0.9747 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 3/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 82s 177ms/step - accuracy: 0.9772 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 4/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 81s 176ms/step - accuracy: 0.9768 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 5/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 83s 184ms/step - accuracy: 0.9753 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 6/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 80s 176ms/step - accuracy: 0.9753 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 7/100
453/453 ━━━━━━━━━━━━━━━━━━━━ 80s 176ms/step - accuracy: 0.9755 - loss: nan - val_accuracy: 0.9762 - val_loss: nan
Epoch 8/100
453/453 ━━━━━━━━━━━━━━━━━━━━

In [ ]:
y_proba = model.predict(X_test).ravel()
y_pred = (y_proba >= 0.5).astype(int)

print("TFT-style AUC:", roc_auc_score(y_test, y_proba))
print("\nОтчёт по классификации (TFT-style):")
print(classification_report(y_test, y_pred, digits=3))
print("Матрица ошибок (TFT-style):")
print(confusion_matrix(y_test, y_pred))


225/225 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step


ValueError: Input contains NaN.